# Create Simplified Binary Labels

This notebook converts the detailed 2010–2018 label file into a simplified
binary label file for model training.

Binary definition:

- `0` = NF: no M1.0-or-greater flare in the following 24 hours
- `1` = FL: at least one M1.0-or-greater flare in the following 24 hours

The original detailed label file will not be modified.

In [1]:
from pathlib import Path

import pandas as pd


In [2]:
# Find the repository root whether the notebook is launched from
# the repository root or from data_labeling/eda.
def find_project_root(start_path):
    for folder in [start_path, *start_path.parents]:
        if (folder / "data_labeling").is_dir() and (folder / "modeling").is_dir():
            return folder

    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())


In [3]:
PROJECT_ROOT

PosixPath('/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-')

In [5]:
INPUT_FILE = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "labels_2010_2018.csv"
)

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "simplified_data_labels"
)

OUTPUT_FILE = OUTPUT_FOLDER / "labels_2010_2018_binary.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("Output file:", OUTPUT_FILE)

Project root: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Input file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/labels_2010_2018.csv
Output file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2010_2018_binary.csv


In [5]:
labels = pd.read_csv(INPUT_FILE)

print("Number of rows:", len(labels))
print("Columns:", labels.columns.tolist())

labels.head()

Number of rows: 63285
Columns: ['label', 'goes_class', 'fl_lon', 'fl_lat', 'rest_fl', 'rest_lon', 'rest_lat']


,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],['-38.5479308163'],['16.0']
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],['-38.5479308163'],['16.0']
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],['-38.5479308163'],['16.0']
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],['-38.5479308163'],['16.0']


In [6]:
required_columns = {"label", "goes_class"}
missing_columns = required_columns - set(labels.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if labels["label"].isna().any():
    raise ValueError("Some image paths are missing.")

if labels["goes_class"].isna().any():
    raise ValueError("Some GOES classes are missing.")

print("Required columns and values are present.")

Required columns and values are present.


In [7]:
goes_values = labels["goes_class"].astype(str).str.strip().str.upper()

class_letters = goes_values.str[0]
class_letters = class_letters.where(goes_values != "NF", "NF")

class_distribution = (
    class_letters
    .value_counts()
    .reindex(["NF", "A", "B", "C", "M", "X"], fill_value=0)
    .rename_axis("GOES class")
    .to_frame("Number of images")
)

class_distribution

,Number of images
GOES class,
NF,12235
A,766
B,16225
C,25084
M,8111
X,864


In [8]:
simplified_labels = labels[["label"]].copy()

# M-class and X-class prediction windows are FL = 1.
# NF, A-class, B-class, and C-class prediction windows are NF = 0.
simplified_labels["goes_class"] = (
    goes_values.str.startswith(("M", "X")).astype(int)
)

simplified_labels.head()

,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0


In [9]:
assert len(simplified_labels) == len(labels)
assert simplified_labels["label"].notna().all()
assert simplified_labels["goes_class"].isin([0, 1]).all()
assert not simplified_labels["label"].duplicated().any()

binary_counts = (
    simplified_labels["goes_class"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename(index={0: "NF (0)", 1: "FL (1)"})
    .rename_axis("Binary class")
    .to_frame("Number of images")
)

binary_counts

,Number of images
Binary class,
NF (0),54310
FL (1),8975


In [10]:
nf_count = int((simplified_labels["goes_class"] == 0).sum())
fl_count = int((simplified_labels["goes_class"] == 1).sum())
nf_to_fl_ratio = nf_count / fl_count

print(f"Total images: {len(simplified_labels):,}")
print(f"NF images:    {nf_count:,}")
print(f"FL images:    {fl_count:,}")
print(f"NF:FL ratio:  {nf_to_fl_ratio:.2f}:1")
print(f"FL:NF ratio:  1:{nf_to_fl_ratio:.2f}")

Total images: 63,285
NF images:    54,310
FL images:    8,975
NF:FL ratio:  6.05:1
FL:NF ratio:  1:6.05


In [11]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

simplified_labels.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Created:", OUTPUT_FILE)
print("Rows saved:", len(simplified_labels))

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2010_2018_binary.csv
Rows saved: 63285


In [12]:
saved_labels = pd.read_csv(OUTPUT_FILE)

assert list(saved_labels.columns) == ["label", "goes_class"]
assert len(saved_labels) == len(simplified_labels)
assert saved_labels["goes_class"].isin([0, 1]).all()
assert saved_labels.equals(simplified_labels)

print("Saved file verified successfully.")
print("Rows:", len(saved_labels))
print()
print(saved_labels["goes_class"].value_counts().sort_index())

saved_labels.head()

Saved file verified successfully.
Rows: 63285

goes_class
0    54310
1     8975
Name: count, dtype: int64


,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0


# Now for data from 2019 to 2026 july

In [6]:
INPUT_FILE = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "labels_2019_2026.csv"
)

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "simplified_data_labels"
)

OUTPUT_FILE = OUTPUT_FOLDER / "labels_2019_2026_binary.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("Output file:", OUTPUT_FILE)

Project root: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Input file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/labels_2019_2026.csv
Output file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2019_2026_binary.csv


In [7]:
INPUT_FILE_2019_2026 = (
    PROJECT_ROOT
    / "data_labeling"
    / "data_labels"
    / "labels_2019_2026_july.csv"
)

OUTPUT_FILE_2019_2026 = (
    OUTPUT_FOLDER
    / "labels_2019_2026_july_binary.csv"
)

labels_2019_2026 = pd.read_csv(INPUT_FILE_2019_2026)

print("Input file:", INPUT_FILE_2019_2026)
print("Output file:", OUTPUT_FILE_2019_2026)
print("Number of rows:", len(labels_2019_2026))
print("Columns:", labels_2019_2026.columns.tolist())

labels_2019_2026.head()

Input file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/labels_2019_2026_july.csv
Output file: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2019_2026_july_binary.csv
Number of rows: 64808
Columns: ['label', 'goes_class', 'fl_lon', 'fl_lat', 'rest_fl', 'rest_lon', 'rest_lat']


,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat
0,2019/01/01/HMI.m2019.01.01_00.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
1,2019/01/01/HMI.m2019.01.01_01.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
2,2019/01/01/HMI.m2019.01.01_02.00.00.jpg,B2.8,21,9,NaN,NaN,NaN
3,2019/01/01/HMI.m2019.01.01_03.00.00.jpg,B2.8,21,9,NaN,NaN,NaN
4,2019/01/01/HMI.m2019.01.01_04.00.00.jpg,B2.8,21,9,NaN,NaN,NaN


In [8]:
required_columns = {"label", "goes_class"}
missing_columns = required_columns - set(labels_2019_2026.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if labels_2019_2026["label"].isna().any():
    raise ValueError("Some image paths are missing.")

if labels_2019_2026["goes_class"].isna().any():
    raise ValueError("Some GOES classes are missing.")

if labels_2019_2026["label"].duplicated().any():
    raise ValueError("Duplicate image paths were found.")

print("The detailed 2019–July 2026 labels passed validation.")

The detailed 2019–July 2026 labels passed validation.


In [9]:
goes_values_2019_2026 = (
    labels_2019_2026["goes_class"]
    .astype(str)
    .str.strip()
    .str.upper()
)

class_letters_2019_2026 = goes_values_2019_2026.str[0]
class_letters_2019_2026 = class_letters_2019_2026.where(
    goes_values_2019_2026 != "NF",
    "NF"
)

class_distribution_2019_2026 = (
    class_letters_2019_2026
    .value_counts()
    .reindex(["NF", "A", "B", "C", "M", "X"], fill_value=0)
    .rename_axis("GOES class")
    .to_frame("Number of images")
)

class_distribution_2019_2026

,Number of images
GOES class,
NF,11320
A,5792
B,8292
C,23450
M,14177
X,1777


In [10]:
simplified_labels_2019_2026 = labels_2019_2026[["label"]].copy()

# M and X become flare (1).
# NF, A, B, and C become non-flare (0).
simplified_labels_2019_2026["goes_class"] = (
    goes_values_2019_2026
    .str.startswith(("M", "X"))
    .astype(int)
)

simplified_labels_2019_2026.head()

,label,goes_class
0,2019/01/01/HMI.m2019.01.01_00.00.00.jpg,0
1,2019/01/01/HMI.m2019.01.01_01.00.00.jpg,0
2,2019/01/01/HMI.m2019.01.01_02.00.00.jpg,0
3,2019/01/01/HMI.m2019.01.01_03.00.00.jpg,0
4,2019/01/01/HMI.m2019.01.01_04.00.00.jpg,0


In [11]:
assert len(simplified_labels_2019_2026) == len(labels_2019_2026)
assert simplified_labels_2019_2026["label"].notna().all()
assert simplified_labels_2019_2026["goes_class"].isin([0, 1]).all()
assert not simplified_labels_2019_2026["label"].duplicated().any()

nf_count_2019_2026 = int(
    (simplified_labels_2019_2026["goes_class"] == 0).sum()
)

fl_count_2019_2026 = int(
    (simplified_labels_2019_2026["goes_class"] == 1).sum()
)

ratio_2019_2026 = nf_count_2019_2026 / fl_count_2019_2026

print(f"Total images: {len(simplified_labels_2019_2026):,}")
print(f"NF images:    {nf_count_2019_2026:,}")
print(f"FL images:    {fl_count_2019_2026:,}")
print(f"NF:FL ratio:  {ratio_2019_2026:.2f}:1")
print(f"FL:NF ratio:  1:{ratio_2019_2026:.2f}")

simplified_labels_2019_2026["goes_class"].value_counts().sort_index()

Total images: 64,808
NF images:    48,854
FL images:    15,954
NF:FL ratio:  3.06:1
FL:NF ratio:  1:3.06


goes_class
0    48854
1    15954
Name: count, dtype: int64

In [12]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

simplified_labels_2019_2026.to_csv(
    OUTPUT_FILE_2019_2026,
    index=False
)

print("Created:", OUTPUT_FILE_2019_2026)
print("Rows saved:", len(simplified_labels_2019_2026))

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/simplified_data_labels/labels_2019_2026_july_binary.csv
Rows saved: 64808


In [13]:
saved_labels_2019_2026 = pd.read_csv(OUTPUT_FILE_2019_2026)

assert list(saved_labels_2019_2026.columns) == [
    "label",
    "goes_class",
]

assert len(saved_labels_2019_2026) == len(
    simplified_labels_2019_2026
)

assert saved_labels_2019_2026["goes_class"].isin([0, 1]).all()

assert saved_labels_2019_2026.equals(
    simplified_labels_2019_2026
)

print("Saved file verified successfully.")
print("Rows:", len(saved_labels_2019_2026))
print()
print(
    saved_labels_2019_2026["goes_class"]
    .value_counts()
    .sort_index()
)

saved_labels_2019_2026.head()

Saved file verified successfully.
Rows: 64808

goes_class
0    48854
1    15954
Name: count, dtype: int64


,label,goes_class
0,2019/01/01/HMI.m2019.01.01_00.00.00.jpg,0
1,2019/01/01/HMI.m2019.01.01_01.00.00.jpg,0
2,2019/01/01/HMI.m2019.01.01_02.00.00.jpg,0
3,2019/01/01/HMI.m2019.01.01_03.00.00.jpg,0
4,2019/01/01/HMI.m2019.01.01_04.00.00.jpg,0
